# Turn-by-Turn Progression Analysis

Agrega os arquivos `turn_states.csv` salvos por cada torneio e plota a **progressão média por rodada**:
recursos (Sol/Plâncton), produção, recursos usados, score, ocupação por camada do board e o ecossistema
(temperatura, cartas restantes).

Cada linha de `turn_states.csv` é um snapshot `(jogo, turno, jogador)`. Aqui reduzimos a **fim-de-rodada**
(último turno de cada rodada) e tiramos a média entre os jogos.

> Nota: rodadas avançadas têm menos jogos (partidas terminam em rodadas diferentes), então as caudas são
> médias apenas dos jogos "sobreviventes" — leia-as com cautela.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
TURN_STATES_PATH = None  # Opcional: aponte para um turn_states.csv específico.


def find_latest_turn_states_csv(root: Path) -> Path:
    base = root / 'artifacts' / 'tournaments'
    dirs = sorted(base.glob('*/'), key=lambda p: p.stat().st_mtime, reverse=True)
    for d in dirs:
        candidate = d / 'turn_states.csv'
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        'Nenhum turn_states.csv encontrado em artifacts/tournaments/. '
        'Rode um torneio primeiro: python scripts/run_tournament.py --games 100'
    )


turn_states_path = Path(TURN_STATES_PATH) if TURN_STATES_PATH else find_latest_turn_states_csv(ROOT)
print('Usando:', turn_states_path)

In [ ]:
df = pd.read_csv(turn_states_path)
print('linhas:', len(df), '| jogos:', df['seed'].nunique(), '| jogadores:', sorted(df['player'].unique()))
print('rodadas:', int(df['round'].min()), '->', int(df['round'].max()))
df.head()

## Agregação: estado de fim-de-rodada, média entre jogos

In [ ]:
# Fim-de-rodada: para cada (jogo, jogador, rodada) pega o snapshot de maior turno.
end_of_round = df.loc[df.groupby(['seed', 'player', 'round'])['turn'].idxmax()]

# Média entre jogos por (rodada, jogador).
agg = end_of_round.groupby(['round', 'player']).mean(numeric_only=True).reset_index()

# Quantos jogos ainda estão vivos em cada rodada (para ler as caudas).
games_alive = end_of_round.groupby('round')['seed'].nunique()

players = sorted(agg['player'].unique())


def series(metric, player):
    sub = agg[agg['player'] == player].set_index('round')[metric].sort_index()
    return sub.index, sub.values


agg.head()

## Recursos por jogador (Sol e Plâncton)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for p in players:
    x, y = series('sun', p)
    ax.plot(x, y, marker='o', label=f'P{p} Sol')
for p in players:
    x, y = series('plankton', p)
    ax.plot(x, y, marker='x', linestyle='--', label=f'P{p} Plâncton')
ax.set_title('Recursos médios por rodada (fim-de-rodada)')
ax.set_xlabel('rodada'); ax.set_ylabel('recurso')
ax.legend(); plt.tight_layout(); plt.show()

## Score por jogador

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for p in players:
    x, y = series('score', p)
    ax.plot(x, y, marker='o', label=f'P{p}')
ax.set_title('Score médio por rodada')
ax.set_xlabel('rodada'); ax.set_ylabel('score')
ax.legend(); plt.tight_layout(); plt.show()

## Recursos usados vs produzidos (acumulado)

Se a produção ficar colada em zero, a economia depende só dos recursos iniciais.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for p in players:
    x, y = series('spent_sun', p); axes[0].plot(x, y, marker='o', label=f'P{p} usado')
    x, y = series('produced_sun', p); axes[0].plot(x, y, linestyle='--', marker='.', label=f'P{p} produzido')
axes[0].set_title('Sol: usado vs produzido (acumulado)'); axes[0].set_xlabel('rodada'); axes[0].legend()

for p in players:
    x, y = series('spent_plankton', p); axes[1].plot(x, y, marker='o', label=f'P{p} usado')
    x, y = series('produced_plankton', p); axes[1].plot(x, y, linestyle='--', marker='.', label=f'P{p} produzido')
axes[1].set_title('Plâncton: usado vs produzido (acumulado)'); axes[1].set_xlabel('rodada'); axes[1].legend()
plt.tight_layout(); plt.show()

## Ocupação do board por camada

`z0` = fundo, `z1` = meio, `z2` = topo. Mostra o quanto o jogo sobe no eixo vertical ao longo das rodadas.

In [ ]:
fig, axes = plt.subplots(1, len(players), figsize=(7 * len(players), 5), squeeze=False)
for col, p in enumerate(players):
    ax = axes[0][col]
    for z, label in [('corals_z0', 'fundo (z0)'), ('corals_z1', 'meio (z1)'), ('corals_z2', 'topo (z2)')]:
        x, y = series(z, p)
        ax.plot(x, y, marker='o', label=label)
    ax.set_title(f'Corais de P{p} por camada'); ax.set_xlabel('rodada'); ax.set_ylabel('nº de corais'); ax.legend()
plt.tight_layout(); plt.show()

## Solos e mão de cartas (flora) por jogador

Quantos tiles de solo cada jogador tem no board e quantas cartas de flora carrega na mão (teto de 10), rodada a rodada.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for p in players:
    x, y = series('soils_count', p)
    axes[0].plot(x, y, marker='o', label=f'P{p}')
axes[0].set_title('Tiles de solo por jogador'); axes[0].set_xlabel('rodada'); axes[0].set_ylabel('nº de solos'); axes[0].legend()

for p in players:
    x, y = series('hand_size', p)
    axes[1].plot(x, y, marker='o', label=f'P{p}')
axes[1].axhline(10, linestyle=':', color='k', label='teto (10)')
axes[1].set_title('Cartas de flora na mão'); axes[1].set_xlabel('rodada'); axes[1].set_ylabel('cartas'); axes[1].legend()
plt.tight_layout(); plt.show()

## Suprimentos: pilha de solo e deck de flora restantes

Compartilhados entre os jogadores. Quando a pilha de solo zera, não dá mais para comprar solo (e sem solo não se constrói coral).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x, y = series('soil_pile_remaining', players[0])  # compartilhado
ax.plot(x, y, marker='o', color='tab:brown', label='pilha de solo')
x, y = series('flora_deck_remaining', players[0])
ax.plot(x, y, marker='o', color='tab:green', label='deck de flora')
ax.set_title('Suprimentos compartilhados restantes'); ax.set_xlabel('rodada'); ax.set_ylabel('cartas/tiles'); ax.legend()
plt.tight_layout(); plt.show()

## Ecossistema: temperatura e cartas de clima

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x, y = series('temperature', players[0])  # compartilhado entre jogadores
axes[0].plot(x, y, marker='o', color='tab:red')
axes[0].axhline(32.0, linestyle=':', color='k', label='crítico (32°C)')
axes[0].set_title('Temperatura média por rodada'); axes[0].set_xlabel('rodada'); axes[0].set_ylabel('°C'); axes[0].legend()

x, y = series('remaining_climate_cards', players[0])
axes[1].plot(x, y, marker='o', color='tab:blue', label='restantes')
x, y = series('consumed_climate_cards', players[0])
axes[1].plot(x, y, marker='.', color='tab:green', label='consumidas')
axes[1].set_title('Cartas de clima por rodada'); axes[1].set_xlabel('rodada'); axes[1].set_ylabel('nº de cartas'); axes[1].legend()
plt.tight_layout(); plt.show()

## Quantos jogos continuam vivos por rodada

Contexto para ler as caudas dos gráficos: a partir da rodada em que a curva cai, há poucos jogos na média.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(games_alive.index, games_alive.values, color='tab:gray')
ax.set_title('Jogos ainda em andamento por rodada'); ax.set_xlabel('rodada'); ax.set_ylabel('nº de jogos')
plt.tight_layout(); plt.show()

## Tabela: progressão média (P1) por rodada

In [ ]:
cols = ['sun', 'plankton', 'produced_sun', 'spent_sun', 'score', 'soils_count', 'hand_size',
        'corals_z0', 'corals_z1', 'corals_z2', 'temperature', 'occupancy_ratio']
table = agg[agg['player'] == players[0]].set_index('round')[cols].round(2)
table.insert(0, 'jogos_vivos', games_alive)
table